<a href="https://colab.research.google.com/github/itsthanush/10212CS216_NLP_VTU24725_Thanush/blob/main/cricket_prediction_main(usecase).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving deliveries.csv to deliveries.csv
Saving matches.csv to matches.csv


In [ ]:
import pandas as pd

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print(matches.head())
print(deliveries.head())

       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_decision  \
0        Kolkat

In [ ]:
# Total runs per match per team
total_runs = deliveries.groupby(['match_id', 'batting_team'])['total_runs'].sum().reset_index()

# Convert into pivot table
runs_pivot = total_runs.pivot(index='match_id', columns='batting_team', values='total_runs').fillna(0)

runs_pivot.reset_index(inplace=True)

runs_pivot.head()

batting_team,match_id,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Delhi Daredevils,Gujarat Lions,Gujarat Titans,Kings XI Punjab,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,335982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.0,0.0,0.0
1,335983,240.0,0.0,0.0,0.0,0.0,0.0,207.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,335984,0.0,0.0,0.0,132.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,129.0,0.0,0.0,0.0,0.0,0.0
3,335985,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,165.0,0.0,0.0,0.0,0.0,0.0,166.0,0.0,0.0
4,335986,0.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
data = matches.merge(runs_pivot, left_on='id', right_on='match_id')

data.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,82.0,0.0,0.0
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,...,0.0,0.0,0.0,0.0,129.0,0.0,0.0,0.0,0.0,0.0
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,...,0.0,165.0,0.0,0.0,0.0,0.0,0.0,166.0,0.0,0.0
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
data['target'] = (data['winner'] == data['team1']).astype(int)

In [ ]:
data = data[['team1', 'team2', 'toss_winner', 'venue', 'target']].dropna()

In [ ]:
from sklearn.preprocessing import LabelEncoder

team_encoder = LabelEncoder()
toss_encoder = LabelEncoder()
venue_encoder = LabelEncoder()

data['team1'] = team_encoder.fit_transform(data['team1'])
data['team2'] = team_encoder.transform(data['team2'])
data['toss_winner'] = toss_encoder.fit_transform(data['toss_winner'])
data['venue'] = venue_encoder.fit_transform(data['venue'])

In [ ]:
from sklearn.model_selection import train_test_split

X = data[['team1', 'team2', 'toss_winner', 'venue']]
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5296803652968036


In [ ]:
# 🔥 Give real input here
team1 = "Chennai Super Kings"
team2 = "Mumbai Indians"
toss_winner = "Chennai Super Kings"
venue = "MA Chidambaram Stadium"

# Convert to numbers using encoders
team1_enc = team_encoder.transform([team1])[0]
team2_enc = team_encoder.transform([team2])[0]
toss_enc = toss_encoder.transform([toss_winner])[0]
venue_enc = venue_encoder.transform([venue])[0]

import pandas as pd

sample = pd.DataFrame([[team1_enc, team2_enc, toss_enc, venue_enc]],
                      columns=['team1', 'team2', 'toss_winner', 'venue'])

prediction = model.predict(sample)

# Output
if prediction[0] == 1:
    print("Predicted Winner:", team1)
else:
    print("Predicted Winner:", team2)

Predicted Winner: Chennai Super Kings
